In [1]:
import sys

model_path = '/Users/rdobrien/skysurf/Zodiacal_Light_Model_Python'
if model_path not in sys.path:
    sys.path.append(model_path)

In [2]:
# # make_baseline.py -- run with the CURRENT get_zmod.py, before updating it
# import numpy as np
# from zodisurf.get_zmod import get_zmod

# N = 6
# DAY = np.linspace(100.0, 300.0, N)
# LON = np.linspace(20.0, 340.0, N)
# LAT = np.linspace(-60.0, 60.0, N)

# out = {}
# for lam in [1.25, 2.2, 3.5, 12.0, 25.0]:          # DIRBE bands
#     out[f'kelsall_{lam}'] = get_zmod(lam, 'kelsall', DAY, LON, LAT)
# for lam in [1.0, 1.25, 2.0, 3.0]:
#     out[f'skysurf_{lam}'] = get_zmod(lam, 'skysurf', DAY, LON, LAT)
# np.savez('zmod_baseline.npz', **out)
# print("saved", list(out))

In [3]:
import numpy as np, warnings
import zodisurf

kw = dict(fitting_hong=[0.7, -0.2, 0.1, 0.6, 0.3, 0.1], fitting_albedo=0.25,
          fitting_em_smooth=0.9, fitting_em_band=0.9, fitting_em_ring=0.9,
          solar_irr=zodisurf.solar_sp(1.25))
n = 300
args = (1.25, 'fitting', np.linspace(100, 300, n),
        np.linspace(20, 340, n), np.linspace(-60, 60, n))

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    z_np = zodisurf.get_zmod(*args, engine='numpy', **kw)
    z_nb = zodisurf.get_zmod(*args, engine='numba', **kw)
print("fallback warnings:", [str(x.message) for x in w])
print("max rel diff:", np.max(np.abs(z_nb / z_np - 1)))

fallback warnings: []
max rel diff: 2.220446049250313e-16


In [4]:
import importlib, numpy as np
import zodisurf
gz = importlib.import_module("zodisurf.get_zmod")
from zodisurf.skysurf_params import get_albedo, get_hong_params, get_mult, get_emiss

LAM = 1.25
lam = np.array([LAM])

alb  = float(np.ravel(get_albedo(lam))[0])
hong = np.array(get_hong_params(lam), dtype=float).reshape(1, -1).copy()
hong[:, -3:] = hong[:, 3:] * get_mult(lam)
em_tab = np.ravel(get_emiss(lam))[0]

zp = gz._default_zpar()
# Emissivity slots: smooth 37, bands 63/89/115/141, ring 174
em_sm, em_bd, em_rg = zp[37], zp[[63, 89, 115, 141]], zp[174]
print("get_mult:", np.ravel(get_mult(lam)), " get_emiss:", em_tab)
print("zpar emiss -> smooth:", em_sm, " bands:", em_bd, " ring:", em_rg)

# If get_emiss is NaN, put_zpar keeps the zpar defaults, so use those
if np.isnan(em_tab):
    em_smooth, em_band, em_ring = float(em_sm), float(em_bd[0]), float(em_rg)
    assert np.allclose(em_bd, em_bd[0]), "band defaults differ; pass them per component"
else:
    em_smooth = em_band = em_ring = float(em_tab)

n = 300
geo = (np.linspace(100, 300, n), np.linspace(20, 340, n), np.linspace(-60, 60, n))
sirr = float(np.ravel(zodisurf.solar_sp(LAM))[0])

z_sky = zodisurf.get_zmod(LAM, 'skysurf', *geo, solar_irr=np.full(n, sirr))
z_fit = zodisurf.get_zmod(LAM, 'fitting', *geo, solar_irr=sirr,
                          fitting_hong=hong[0], fitting_albedo=alb,
                          fitting_em_smooth=em_smooth, fitting_em_band=em_band,
                          fitting_em_ring=em_ring)

print("max abs diff:", np.max(np.abs(z_fit - z_sky)))
print("match:", np.allclose(z_fit, z_sky, rtol=0, atol=6e-6))

get_mult: [1.]  get_emiss: nan
zpar emiss -> smooth: 1.659892404064974  bands: [1.6598924 1.6598924 1.6598924 1.6598924]  ring: 1.659892404064974
max abs diff: 4.995848786243373e-06
match: True


In [5]:
# ======================================================================
#  Timing harness: 10 Metropolis steps, 100 pointings, 10 free parameters
#  Measures the cost of ONE likelihood evaluation as MCMC would call it.
# ======================================================================
import time
import numpy as np
import zodisurf

LAM     = 1.25      # micron
NPOINT  = 600       # pointings per likelihood call
NSTEP   = 10        # proposals to time
SEED    = 42
ENGINES = ('numpy', 'numba')

rng = np.random.default_rng(SEED)

# ---- geometry: fixed for the whole run, as in a real fit --------------
day = rng.uniform(100.0, 300.0, NPOINT)
lon = rng.uniform(0.0, 360.0, NPOINT)
lat = rng.uniform(-75.0, 75.0, NPOINT)
sirr = float(np.ravel(zodisurf.solar_sp(LAM))[0])

# ---- free parameters: [g1,g2,g3, w1,w2,w3, albedo, em_sm, em_bd, em_rg]
P_NAMES = ['g1', 'g2', 'g3', 'w1', 'w2', 'w3',
           'albedo', 'em_smooth', 'em_band', 'em_ring']
p_true = np.array([0.5150, -0.2218, -0.8704,
                   0.063804, 0.036425, 0.00037770,
                   0.21, 1.6599, 1.6599, 1.6599])
# Proposal widths: ~1% of each value (tune to your real step sizes)
p_step = np.abs(p_true) * 0.01


def model(p, engine):
    """One likelihood-sized call to get_zmod in fitting mode."""
    return zodisurf.get_zmod(
        LAM, 'fitting', day, lon, lat, solar_irr=sirr,
        fitting_hong=p[0:6], fitting_albedo=p[6],
        fitting_em_smooth=p[7], fitting_em_band=p[8], fitting_em_ring=p[9],
        engine=engine, n_jobs=1)          # n_jobs=1: let the sampler parallelise


def log_prior(p):
    g, w = p[0:3], p[3:6]
    if np.any(np.abs(g) >= 1.0):      return -np.inf   # Henyey-Greenstein
    if np.any(w < 0.0):               return -np.inf
    if not (0.0 <= p[6] <= 1.0):      return -np.inf   # albedo
    if np.any(p[7:10] <= 0.0):        return -np.inf   # emissivity
    return 0.0


# ---- synthetic data: truth + 5% noise --------------------------------
data = model(p_true, 'numpy')
sigma = 0.05 * np.abs(data)
data = data + rng.normal(0.0, sigma)


def log_like(p, engine):
    lp = log_prior(p)
    if not np.isfinite(lp):
        return -np.inf
    return lp - 0.5 * np.sum(((model(p, engine) - data) / sigma) ** 2)


# ---- run ------------------------------------------------------------
print(f"lambda={LAM} um, {NPOINT} pointings, {len(p_true)} free parameters, "
      f"{NSTEP} steps\n")

for engine in ENGINES:
    # Warm-up excluded from the timing: numba JIT-compiles on first call
    t0 = time.perf_counter()
    _ = log_like(p_true, engine)
    t_warm = time.perf_counter() - t0

    p, lnp = p_true.copy(), log_like(p_true, engine)
    times, naccept = [], 0

    for i in range(NSTEP):
        q = p + p_step * rng.standard_normal(p.size)
        t0 = time.perf_counter()
        lnq = log_like(q, engine)
        times.append(time.perf_counter() - t0)
        if np.isfinite(lnq) and np.log(rng.random()) < lnq - lnp:
            p, lnp = q, lnq
            naccept += 1

    t = np.array(times) * 1e3        # ms
    print(f"{engine:>6}:  median {np.median(t):8.2f} ms   "
          f"mean {t.mean():8.2f} ms   min {t.min():8.2f} ms   "
          f"max {t.max():8.2f} ms")
    print(f"         first call (incl. JIT) {t_warm*1e3:.1f} ms,  "
          f"accepted {naccept}/{NSTEP}")

print("\nPer-step cost in a real run = (this time) x nwalkers / n_parallel_procs")

lambda=1.25 um, 600 pointings, 10 free parameters, 10 steps

 numpy:  median    43.06 ms   mean    43.70 ms   min    42.55 ms   max    46.51 ms
         first call (incl. JIT) 42.9 ms,  accepted 0/10
 numba:  median    22.27 ms   mean    22.20 ms   min    21.90 ms   max    22.48 ms
         first call (incl. JIT) 23.1 ms,  accepted 1/10

Per-step cost in a real run = (this time) x nwalkers / n_parallel_procs


In [ ]:
import numpy as np, zodisurf
zp = zodisurf.get_zmod.__globals__['_default_zpar']()
zp[11] = 1.0                      # To2 != 0 -> numba guard trips
try:
    zodisurf.get_zmod(1.25, 'skysurf', 150.0, 90.0, 30.0,
                      solar_irr=float(np.ravel(zodisurf.solar_sp(1.25))[0]),
                      zpar=zp, engine='numba')
    print("NO ERROR - fallback is still in place")
except RuntimeError as e:
    print("raised as expected:", e)